# Attention Bottleneck Analysis

Identify information bottlenecks in attention: rank, throughput,
concentration, layer-level, and position-level bottlenecks.

## Why This Matters

Attention bottleneck reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_bottleneck_analysis import (
    attention_rank_bottleneck, attention_information_throughput,
    attention_source_concentration, attention_layer_bottleneck,
    attention_position_bottleneck,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Rank Bottleneck

Low effective rank = head collapses to attending to few positions.

In [ ]:
result = attention_rank_bottleneck(model, tokens)
print(f"Bottleneck heads: {result['n_bottleneck']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    flag = ' [BOTTLENECK]' if h['is_bottleneck'] else ''
    bar = '█' * int(min(h['effective_rank'], 5) * 3)
    print(f"  L{h['layer']}H{h['head']}: rank={h['effective_rank']:.2f}, "
          f"top_sv={h['top_sv_fraction']:.3f}{flag} {bar}")

## Information Throughput

How much value information flows through each head?

In [ ]:
result = attention_information_throughput(model, tokens)
for h in result['per_head']:
    bar = '█' * int(min(h['mean_throughput'] * 20, 20))
    print(f"  L{h['layer']}H{h['head']}: mean={h['mean_throughput']:.4f}, "
          f"max={h['max_throughput']:.4f}, v_norm={h['mean_value_norm']:.4f} {bar}")

## Source Concentration

Heads with high concentration attend almost exclusively to one source.

In [ ]:
result = attention_source_concentration(model, tokens)
print(f"Concentrated heads: {result['n_concentrated']}\n")
for h in result['per_head']:
    flag = ' [CONCENTRATED]' if h['is_concentrated'] else ''
    print(f"  L{h['layer']}H{h['head']}: entropy={h['mean_entropy']:.4f}, "
          f"norm_H={h['normalized_entropy']:.3f}, "
          f"top1={h['mean_top1_mass']:.3f}{flag}")

## Layer Bottleneck

Per-layer summary of attention bottleneck severity.

In [ ]:
result = attention_layer_bottleneck(model, tokens)
for p in result['per_layer']:
    flag = ' [HAS BOTTLENECK]' if p['has_bottleneck_head'] else ''
    print(f"  Layer {p['layer']}: mean_rank={p['mean_pattern_rank']:.2f}, "
          f"min_rank={p['min_pattern_rank']:.2f}, "
          f"entropy={p['mean_attn_entropy']:.4f}{flag}")

## Position Bottleneck

Which query positions receive information from very few sources?

In [ ]:
result = attention_position_bottleneck(model, tokens)
print(f"Bottlenecked positions: {result['n_bottlenecked']}/{len(result['per_position'])}\n")
for p in result['per_position']:
    flag = ' [BOTTLENECK]' if p['is_bottlenecked'] else ''
    bar = '█' * int(min(p['mean_entropy'] * 5, 15))
    print(f"  Pos {p['position']}: entropy={p['mean_entropy']:.4f}, "
          f"top1={p['mean_top1_mass']:.3f}{flag} {bar}")